<a href="https://colab.research.google.com/github/imdann06/diabetes-risk-prediction/blob/develop/notebooks/Preprocesamiento_Diabetes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Limpieza, Preprocesamiento de Datos e Imputación

**Proyecto:** Diabetes Risk Prediction  
**Responsable:** Daniel

En este notebook se realiza el diagnóstico, limpieza y preprocesamiento
del conjunto de datos de predicción de riesgo de diabetes.

In [ ]:
# Importar Librerias

In [4]:
import pandas as pd
import numpy as np
import zipfile
import os
import glob

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import joblib

#Subir y cargar el dataset

In [5]:
from google.colab import files

uploaded = files.upload()

zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall("/content/diabetes_dataset")

csv_files = glob.glob(
    "/content/diabetes_dataset/**/*.csv",
    recursive=True
)

csv_path = csv_files[0]

df = pd.read_csv(csv_path)

print("Archivo cargado:", csv_path)
print("Dimensiones del dataset:", df.shape)

Saving diabetes_risk_prediction_dataset.zip.zip to diabetes_risk_prediction_dataset.zip (1).zip
Archivo cargado: /content/diabetes_dataset/diabetes_risk_prediction_dataset.csv
Dimensiones del dataset: (50000, 41)


#Exploración inicial

In [6]:
print("Primeras 5 filas:")
display(df.head())

print("\nDimensiones:")
print(df.shape)

print("\nNombres de las variables:")
print(df.columns.tolist())

print("\nTipos de datos:")
display(df.dtypes.to_frame("Tipo"))

Primeras 5 filas:


,Patient_ID,Age,Gender,Country,Height_cm,Weight_kg,BMI,Waist_Circumference_cm,Blood_Glucose,HbA1c,...,Fatty_Liver,PCOS,Medication_Adherence,Work_Type,Residence_Type,Daily_Water_Intake_L,Diabetes_Risk_Score,AI_Health_Recommendation,Doctor_Consultation_Needed,Diabetes_Risk
0,1,32.0,Male,Mexico,182.1,65.8,19.8,71.2,88.4,10.4,...,Yes,No,Good,Retired,Rural,4.4,60,Healthy Diet Plan,Yes,Moderate
1,2,42.0,Other,Saudi Arabia,148.5,101.2,45.9,121.8,247.3,11.3,...,Yes,Yes,Average,Government,Rural,2.1,99,Begin Diabetes Management Plan,Yes,High
2,3,89.0,Other,Argentina,158.1,NaN,37.9,131.8,141.9,6.3,...,Yes,No,Average,Government,Urban,2.8,80,Strict Blood Sugar Monitoring,Yes,High
3,4,87.0,Other,United States,190.9,95.9,26.3,99.1,90.1,7.4,...,No,No,Average,Business,Rural,1.5,82,Begin Diabetes Management Plan,Yes,High
4,5,27.0,Other,Australia,156.9,101.9,41.4,77.1,93.8,12.0,...,Yes,Yes,Poor,Private,Rural,1.6,93,Begin Diabetes Management Plan,Yes,High



Dimensiones:
(50000, 41)

Nombres de las variables:
['Patient_ID', 'Age', 'Gender', 'Country', 'Height_cm', 'Weight_kg', 'BMI', 'Waist_Circumference_cm', 'Blood_Glucose', 'HbA1c', 'Fasting_Blood_Sugar', 'Insulin_Level', 'Blood_Pressure_Systolic', 'Blood_Pressure_Diastolic', 'Total_Cholesterol', 'HDL', 'LDL', 'Triglycerides', 'Heart_Rate', 'Physical_Activity_Level', 'Exercise_Hours_Per_Week', 'Daily_Walking_Minutes', 'Diet_Quality', 'Sugar_Intake_Level', 'Sleep_Hours', 'Stress_Level', 'Smoking_Status', 'Alcohol_Consumption', 'Family_History_Diabetes', 'Hypertension', 'Heart_Disease', 'Fatty_Liver', 'PCOS', 'Medication_Adherence', 'Work_Type', 'Residence_Type', 'Daily_Water_Intake_L', 'Diabetes_Risk_Score', 'AI_Health_Recommendation', 'Doctor_Consultation_Needed', 'Diabetes_Risk']

Tipos de datos:


,Tipo
Patient_ID,int64
Age,float64
Gender,object
Country,object
Height_cm,float64
Weight_kg,float64
BMI,float64
Waist_Circumference_cm,float64
Blood_Glucose,float64
HbA1c,float64


#Estadísticas generales



In [7]:
display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Patient_ID,50000.0,NaN,NaN,NaN,25000.5,14433.901067,1.0,12500.75,25000.5,37500.25,50000.0
Age,49503.0,NaN,NaN,NaN,53.871099,21.130148,18.0,35.0,54.0,72.0,90.0
Gender,50000,3,Male,16718,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,50000,25,Malaysia,2083,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Height_cm,47055.0,NaN,NaN,NaN,170.018814,14.457787,145.0,157.6,170.0,182.6,195.0
Weight_kg,47038.0,NaN,NaN,NaN,87.457003,24.500358,45.0,66.3,87.3,108.6,130.0
BMI,50000.0,NaN,NaN,NaN,30.924056,10.290091,11.9,22.7,30.0,37.8,61.7
Waist_Circumference_cm,50000.0,NaN,NaN,NaN,99.96079,23.086579,60.0,80.0,99.8,119.9,140.0
Blood_Glucose,49041.0,NaN,NaN,NaN,159.905245,52.040575,70.0,114.7,159.9,205.1,250.0
HbA1c,48026.0,NaN,NaN,NaN,8.488061,2.313758,4.5,6.5,8.5,10.5,12.5


#Valores faltantes

In [8]:
faltantes = pd.DataFrame({
    "Valores faltantes": df.isnull().sum(),
    "Porcentaje (%)": (df.isnull().mean() * 100).round(2)
})

faltantes = faltantes.sort_values(
    by="Valores faltantes",
    ascending=False
)

display(faltantes[faltantes["Valores faltantes"] > 0])

print(
    "\nTotal de valores faltantes:",
    int(df.isnull().sum().sum())
)

,Valores faltantes,Porcentaje (%)
Weight_kg,2962,5.92
Height_cm,2945,5.89
Sleep_Hours,2017,4.03
Exercise_Hours_Per_Week,1992,3.98
HbA1c,1974,3.95
Daily_Walking_Minutes,1964,3.93
HDL,1000,2.00
LDL,1000,2.00
Triglycerides,1000,2.00
Medication_Adherence,1000,2.00



Total de valores faltantes: 20810


#Revisar duplicados

In [9]:
duplicados = df.duplicated().sum()

print("Registros duplicados:", duplicados)

Registros duplicados: 0


#Separar variable objetivo

In [10]:
TARGET = "Diabetes_Risk"

# Columnas que no utilizaremos como características
columnas_excluir = [
    "Patient_ID",
    "Diabetes_Risk_Score",
    "AI_Health_Recommendation",
    "Doctor_Consultation_Needed"
]

columnas_excluir = [
    col for col in columnas_excluir
    if col in df.columns
]

X = df.drop(columns=[TARGET] + columnas_excluir)
y = df[TARGET]

print("Variable objetivo:", TARGET)
print("Columnas excluidas:", columnas_excluir)

print("\nDimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

Variable objetivo: Diabetes_Risk
Columnas excluidas: ['Patient_ID', 'Diabetes_Risk_Score', 'AI_Health_Recommendation', 'Doctor_Consultation_Needed']

Dimensiones de X: (50000, 36)
Dimensiones de y: (50000,)


#Revisar la variable objetivo

In [11]:
print("Distribución de Diabetes_Risk:")

display(
    y.value_counts()
    .to_frame("Cantidad")
)

print("\nPorcentaje:")

display(
    (y.value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("Porcentaje")
)

Distribución de Diabetes_Risk:


,Cantidad
Diabetes_Risk,
High,36593
Moderate,12937
Low,470



Porcentaje:


,Porcentaje
Diabetes_Risk,
High,73.19
Moderate,25.87
Low,0.94


#Identificar la variable numericas y categoricas

In [12]:
numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("VARIABLES NUMÉRICAS")
print(numeric_features)

print("\nCantidad:", len(numeric_features))

print("\nVARIABLES CATEGÓRICAS")
print(categorical_features)

print("\nCantidad:", len(categorical_features))

VARIABLES NUMÉRICAS
['Age', 'Height_cm', 'Weight_kg', 'BMI', 'Waist_Circumference_cm', 'Blood_Glucose', 'HbA1c', 'Fasting_Blood_Sugar', 'Insulin_Level', 'Blood_Pressure_Systolic', 'Blood_Pressure_Diastolic', 'Total_Cholesterol', 'HDL', 'LDL', 'Triglycerides', 'Heart_Rate', 'Exercise_Hours_Per_Week', 'Daily_Walking_Minutes', 'Sleep_Hours', 'Daily_Water_Intake_L']

Cantidad: 20

VARIABLES CATEGÓRICAS
['Gender', 'Country', 'Physical_Activity_Level', 'Diet_Quality', 'Sugar_Intake_Level', 'Stress_Level', 'Smoking_Status', 'Alcohol_Consumption', 'Family_History_Diabetes', 'Hypertension', 'Heart_Disease', 'Fatty_Liver', 'PCOS', 'Medication_Adherence', 'Work_Type', 'Residence_Type']

Cantidad: 16


#Separar entrenamiento y prueba

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)

print("\nObjetivo entrenamiento:", y_train.shape)
print("Objetivo prueba:", y_test.shape)

Datos de entrenamiento: (40000, 36)
Datos de prueba: (10000, 36)

Objetivo entrenamiento: (40000,)
Objetivo prueba: (10000,)


#Pipeline para variables numéricas

In [17]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

#Pipeline para variables categóricas

In [18]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

#Crear el preprocesador completo

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

#Aplicar el preprocesamiento

In [20]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("X_train original:", X_train.shape)
print("X_train procesado:", X_train_processed.shape)

print("\nX_test original:", X_test.shape)
print("X_test procesado:", X_test_processed.shape)

X_train original: (40000, 36)
X_train procesado: (40000, 86)

X_test original: (10000, 36)
X_test procesado: (10000, 86)


#Comprobar que no quedaron valores faltantes

In [21]:
print(
    "NaN en entrenamiento:",
    np.isnan(X_train_processed).sum()
)

print(
    "NaN en prueba:",
    np.isnan(X_test_processed).sum()
)

NaN en entrenamiento: 0
NaN en prueba: 0


#Obtener nombres de las nuevas variables

In [22]:
feature_names = preprocessor.get_feature_names_out()

print("Número de variables después del procesamiento:")
print(len(feature_names))

print("\nPrimeras variables:")
print(feature_names[:20])

Número de variables después del procesamiento:
86

Primeras variables:
['num__Age' 'num__Height_cm' 'num__Weight_kg' 'num__BMI'
 'num__Waist_Circumference_cm' 'num__Blood_Glucose' 'num__HbA1c'
 'num__Fasting_Blood_Sugar' 'num__Insulin_Level'
 'num__Blood_Pressure_Systolic' 'num__Blood_Pressure_Diastolic'
 'num__Total_Cholesterol' 'num__HDL' 'num__LDL' 'num__Triglycerides'
 'num__Heart_Rate' 'num__Exercise_Hours_Per_Week'
 'num__Daily_Walking_Minutes' 'num__Sleep_Hours'
 'num__Daily_Water_Intake_L']


#Crear DataFrame procesado

In [23]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

display(X_train_processed_df.head())

,num__Age,num__Height_cm,num__Weight_kg,num__BMI,num__Waist_Circumference_cm,num__Blood_Glucose,num__HbA1c,num__Fasting_Blood_Sugar,num__Insulin_Level,num__Blood_Pressure_Systolic,...,cat__Medication_Adherence_Average,cat__Medication_Adherence_Good,cat__Medication_Adherence_Poor,cat__Work_Type_Business,cat__Work_Type_Government,cat__Work_Type_Private,cat__Work_Type_Retired,cat__Work_Type_Student,cat__Residence_Type_Rural,cat__Residence_Type_Urban
32943,-0.467160,-1.755970,1.131871,2.253401,0.394828,0.139896,0.353464,-1.647581,1.471787,0.132660,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
48323,-1.370651,-1.691800,-0.737736,0.173400,-0.498178,-1.719354,-0.264021,-0.438988,-0.973487,0.475744,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
37019,-0.800025,1.445406,0.769740,-0.166787,-1.399854,-1.624355,-1.763627,0.818843,-1.021908,-0.244732,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
45232,0.103466,0.974825,1.637171,0.630223,-0.199065,-1.397523,1.544328,0.722603,-0.158396,-0.759358,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
48854,-1.608412,-0.108937,0.896065,0.717699,0.247438,-0.899268,0.618100,0.460741,-1.078400,0.578669,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


#Comprobar el escalamiento

In [24]:
scaled_columns = [
    col for col in X_train_processed_df.columns
    if col.startswith("num__")
]

display(
    X_train_processed_df[scaled_columns]
    .agg(["mean", "std"])
    .T
    .head(15)
)

,mean,std
num__Age,-1.469935e-16,1.000013
num__Height_cm,1.519673e-15,1.000013
num__Weight_kg,-4.579448e-16,1.000013
num__BMI,8.313350e-17,1.000013
num__Waist_Circumference_cm,6.240342e-16,1.000013
num__Blood_Glucose,-4.298784e-17,1.000013
num__HbA1c,2.176037e-16,1.000013
num__Fasting_Blood_Sugar,9.876544e-17,1.000013
num__Insulin_Level,-7.283063e-17,1.000013
num__Blood_Pressure_Systolic,2.875922e-16,1.000013


#Crear dataset final de entrenamiento y prueba

In [25]:
train_final = X_train_processed_df.copy()
train_final[TARGET] = y_train.values

test_final = X_test_processed_df.copy()
test_final[TARGET] = y_test.values

print("Dataset final de entrenamiento:", train_final.shape)
print("Dataset final de prueba:", test_final.shape)

display(train_final.head())

Dataset final de entrenamiento: (40000, 87)
Dataset final de prueba: (10000, 87)


,num__Age,num__Height_cm,num__Weight_kg,num__BMI,num__Waist_Circumference_cm,num__Blood_Glucose,num__HbA1c,num__Fasting_Blood_Sugar,num__Insulin_Level,num__Blood_Pressure_Systolic,...,cat__Medication_Adherence_Good,cat__Medication_Adherence_Poor,cat__Work_Type_Business,cat__Work_Type_Government,cat__Work_Type_Private,cat__Work_Type_Retired,cat__Work_Type_Student,cat__Residence_Type_Rural,cat__Residence_Type_Urban,Diabetes_Risk
32943,-0.467160,-1.755970,1.131871,2.253401,0.394828,0.139896,0.353464,-1.647581,1.471787,0.132660,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,High
48323,-1.370651,-1.691800,-0.737736,0.173400,-0.498178,-1.719354,-0.264021,-0.438988,-0.973487,0.475744,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,High
37019,-0.800025,1.445406,0.769740,-0.166787,-1.399854,-1.624355,-1.763627,0.818843,-1.021908,-0.244732,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,Moderate
45232,0.103466,0.974825,1.637171,0.630223,-0.199065,-1.397523,1.544328,0.722603,-0.158396,-0.759358,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,High
48854,-1.608412,-0.108937,0.896065,0.717699,0.247438,-0.899268,0.618100,0.460741,-1.078400,0.578669,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,High


#Validación final

In [26]:
print("========== VALIDACIÓN FINAL ==========")

print("\nRegistros de entrenamiento:", len(train_final))
print("Registros de prueba:", len(test_final))

print(
    "\nValores faltantes en entrenamiento:",
    train_final.drop(columns=[TARGET]).isnull().sum().sum()
)

print(
    "Valores faltantes en prueba:",
    test_final.drop(columns=[TARGET]).isnull().sum().sum()
)

print(
    "\nValores NaN en entrenamiento:",
    np.isnan(
        train_final.drop(columns=[TARGET]).values
    ).sum()
)

print(
    "Valores NaN en prueba:",
    np.isnan(
        test_final.drop(columns=[TARGET]).values
    ).sum()
)

print("\nNúmero de variables finales:", X_train_processed_df.shape[1])

========== VALIDACIÓN FINAL ==========

Registros de entrenamiento: 40000
Registros de prueba: 10000

Valores faltantes en entrenamiento: 0
Valores faltantes en prueba: 0

Valores NaN en entrenamiento: 0
Valores NaN en prueba: 0

Número de variables finales: 86


#Guardar los datasets

In [27]:
train_final.to_csv(
    "/content/train_preprocesado.csv",
    index=False
)

test_final.to_csv(
    "/content/test_preprocesado.csv",
    index=False
)

print("Archivos guardados correctamente.")

Archivos guardados correctamente.


#Guardar el preprocesador

In [28]:
joblib.dump(
    preprocessor,
    "/content/preprocesador_diabetes.pkl"
)

print("Preprocesador guardado correctamente.")

Preprocesador guardado correctamente.


#Descargar los resultados

In [29]:
from google.colab import files

files.download("/content/train_preprocesado.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
files.download("/content/test_preprocesado.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
files.download("/content/preprocesador_diabetes.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>